## Visualize Quality Metrics

Basically we just need to write some scaffolding to download all the metrics from CSD3 and then turn this into a pretty dictionary / table (you do you), to easily visualize. Easy peasy!

In [1]:
import os
import json
from pathlib import Path
from typing import Any

from huggingface_hub import snapshot_download
import matplotlib.pyplot as plt
import pandas as pd

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

/Users/ljvmiranda/Developer/multilingual-teacher-eval/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def download_hf_files(repo_id: str, *, directory_path: str, local_path: Path):
    """Download files from a Hugging Face repository to a local directory."""
    if not local_path.exists():
        local_path.mkdir(parents=True, exist_ok=True)
    data_path = snapshot_download(
        repo_id=repo_id,
        repo_type="dataset",
        local_dir=local_path,
        allow_patterns=[directory_path + "/*"],
    )
    return Path(data_path) / directory_path

In [3]:
metric_files = download_hf_files(
    "ljvmiranda921/mtep-intrinsic-metrics",
    directory_path="csd3",
    local_path=Path("./data/"),
)

Fetching 36 files: 100%|██████████| 36/36 [00:14<00:00,  2.51it/s]


In [4]:
def parse_metrics(metrics_dir: Path) -> list[dict]:
    """Parse metric files and extract language, model, and data.

    Expected filename format: msde-S1-{language}_{model}_intrinsic_metrics
    Example: msde-S1-ar_mistralai__Mistral-Small-24B-Instruct-2501_intrinsic_metrics
    """
    metrics_data = []
    for file in metrics_dir.glob("*.json"):
        filename = file.stem
        parts = filename.replace("msde-S1-", "").replace("_intrinsic_metrics", "")
        language, model_raw = parts.split("_", 1)
        model = model_raw.replace("__", "/")

        with open(file, "r") as f:
            data = json.load(f)

        metrics_data.append(
            # fmt: off
            {
                "language": language,
                "model": model,
                "prompts_distinct_ri": data.get("distinct_ri", {}).get("prompts_distinct_ri"),
                "responses_distinct_ri": data.get("distinct_ri", {}).get("prompts_distinct_ri"),
                "rubric_score": data.get("reward_model", {}).get("average_rubric_score"),
                "perplexity": data.get("perplexity", {}).get("average_perplexity"),
            }
            # fmt: on
        )

    return metrics_data

In [5]:
df = (
    pd.DataFrame(parse_metrics(metric_files))
    .sort_values(by=["language", "model"])
    .reset_index(drop=True)
)
# Just use ar, cs, de for now
df = df[df["language"].isin(["ar", "cs", "de"])].reset_index(drop=True)

df

,language,model,prompts_distinct_ri,responses_distinct_ri,rubric_score,perplexity
0,ar,CohereLabs/aya-expanse-32b,0.692789,0.692789,3.9640,4.337098
1,ar,cohere-command-a,0.690184,0.690184,3.9956,NaN
2,ar,google/gemma-3-12b-it,0.721363,0.721363,3.7736,4.431461
3,ar,google/gemma-3-27b-it,NaN,NaN,3.9315,4.403990
4,ar,google/gemma-3-4b-it,NaN,NaN,3.4699,5.518703
5,ar,gpt-4o-mini-2024-07-18,0.703685,0.703685,3.5159,NaN
6,ar,ibm-granite/granite-4.0-1b,NaN,NaN,2.4633,18604.332290
7,ar,ibm-granite/granite-4.0-micro,0.741429,0.741429,3.0330,12.454267
8,ar,meta-llama/Llama-3.1-70B-Instruct,0.700566,0.700566,2.7185,7.004205
9,ar,meta-llama/Llama-3.1-8B-Instruct,0.707724,0.707724,1.7314,62410.433995
